# Step 2 — Build the Historical Outage Panel

**Objective:** Create a county x day panel of power outage metrics from EAGLE-I (2018-2024).

**Data source:** EAGLE-I Power Outage Data, ORNL — download annual CSVs from OSTI.

**Output:** `data/processed/outage_panel_ercot.csv` and `outage_panel_caiso.csv`

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import PROCESSED, ERCOT_FIPS, CAISO_FIPS
from src.data.eagle_i import build_ercot_panel, build_caiso_panel


## 2.1 Build ERCOT outage panel

Requires EAGLE-I annual CSVs in `data/raw/eagle_i/`. Download from ORNL OSTI (see `scripts/download_data.py`).

In [2]:
try:
    ercot_outage = build_ercot_panel()
    print(f'ERCOT panel shape: {ercot_outage.shape}')
    print(ercot_outage.head())
except FileNotFoundError as e:
    print(f'Data not yet available:\n{e}')
    ercot_outage = None


ERCOT panel shape: (497425, 6)
                  max_customers_out  total_customer_hours  total_customers  \
fips  date                                                                   
48001 2018-01-01                4.0                 21.25          28389.0   
      2018-01-02                4.0                 14.25          28389.0   
      2018-01-03               50.0                 68.00          28389.0   
      2018-01-04               32.0                 28.25          28389.0   
      2018-01-05               40.0                 49.00          28389.0   

                  n_intervals  outage_fraction  outage_event_flag  
fips  date                                                         
48001 2018-01-01           25         0.000141                  0  
      2018-01-02           15         0.000141                  0  
      2018-01-03           22         0.001761                  0  
      2018-01-04           22         0.001127                  0  
      2018-01-

## 2.2 Build CAISO outage panel

In [3]:
try:
    caiso_outage = build_caiso_panel()
    print(f'CAISO panel shape: {caiso_outage.shape}')
    print(caiso_outage.head())
except FileNotFoundError as e:
    print(f'Data not yet available:\n{e}')
    caiso_outage = None


CAISO panel shape: (128042, 6)
                  max_customers_out  total_customer_hours  total_customers  \
fips  date                                                                   
06001 2018-06-01                0.0                  0.00         749404.0   
      2018-06-05              108.0                 53.75         749404.0   
      2018-06-06              395.0               1825.25         749404.0   
      2018-06-07              335.0               1902.50         749404.0   
      2018-06-08              457.0               4484.00         749404.0   

                  n_intervals  outage_fraction  outage_event_flag  
fips  date                                                         
06001 2018-06-01            1         0.000000                  0  
      2018-06-05            2         0.000144                  0  
      2018-06-06           83         0.000527                  0  
      2018-06-07           89         0.000447                  0  
      2018-06-

## 2.3 Data quality check

Flag county-years with low EAGLE-I coverage (< 80 intervals/day).

In [4]:
if ercot_outage is not None:
    print('Outage fraction distribution:')
    print(ercot_outage['outage_fraction'].describe())


Outage fraction distribution:
count    497412.000000
mean          0.008651
std           0.059019
min           0.000000
25%           0.000188
50%           0.000644
75%           0.002657
max          12.470582
Name: outage_fraction, dtype: float64


## 2.4 Summary statistics — event rates by year

In [5]:
if ercot_outage is not None:
    event_rate = ercot_outage['outage_event_flag'].mean()
    print(f'ERCOT outage event rate: {event_rate:.3%}')
    df_reset = ercot_outage.reset_index()
    df_reset['year'] = df_reset['date'].dt.year
    yearly = df_reset.groupby('year')['total_customer_hours'].mean()
    print('\nMean customer-hours per county-day by year:')
    print(yearly)


ERCOT outage event rate: 11.676%

Mean customer-hours per county-day by year:
year
2018     475.769898
2019     929.557151
2020     970.530096
2021    3966.438691
2022     764.679762
2023    1514.173447
2024    4743.196444
Name: total_customer_hours, dtype: float64


## 2.5 Save processed panels

In [6]:
if ercot_outage is not None:
    ercot_outage.to_csv(PROCESSED['outage_ercot'])
    print('Saved:', PROCESSED['outage_ercot'])
if caiso_outage is not None:
    caiso_outage.to_csv(PROCESSED['outage_caiso'])
    print('Saved:', PROCESSED['outage_caiso'])


Saved: /burg-archive/home/mck2199/electric-grid-resilience/data/processed/outage_panel_ercot.csv


Saved: /burg-archive/home/mck2199/electric-grid-resilience/data/processed/outage_panel_caiso.csv
